# Automatic Music Transcription with basic-pitch

**CS 89.02 / MUS 14.05 — Music and AI, Week 4**

[basic-pitch](https://github.com/spotify/basic-pitch) is Spotify's open-source
automatic music transcription (AMT) model. It converts audio to MIDI using a
lightweight neural network, handling:

- Polyphonic transcription (multiple simultaneous notes)
- Pitch bend detection
- Onset and offset estimation

This notebook demonstrates running basic-pitch on audio and visualizing
the resulting piano roll.

In [ ]:
!pip install basic-pitch librosa matplotlib numpy pretty_midi

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

# Load sample audio
audio_path = librosa.example('brahms')
y, sr = librosa.load(audio_path, duration=15)  # first 15 seconds

# Save as wav for basic-pitch (it needs a file path)
import soundfile as sf
wav_path = '/tmp/sample_for_transcription.wav'
sf.write(wav_path, y, sr)

print(f"Audio: {len(y)/sr:.1f} seconds at {sr} Hz")
ipd.Audio(y, rate=sr)

## Automatic Music Transcription

basic-pitch uses a neural network to predict three outputs from the audio:

1. **Note posterior** — probability of each pitch being active at each time frame
2. **Onset posterior** — probability of a note onset at each frame
3. **Pitch contour** — continuous pitch estimates (for pitch bend)

These are combined with thresholding and note-tracking heuristics to produce
a MIDI file.

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

# Run transcription
model_output, midi_data, note_events = predict(
    wav_path,
    onset_threshold=0.5,
    frame_threshold=0.3,
    minimum_note_length=58,  # ms
)

print(f"Transcribed {len(midi_data.instruments[0].notes)} notes")
print(f"Instruments: {len(midi_data.instruments)}")

# Save MIDI output
midi_output_path = '/tmp/transcribed.mid'
midi_data.write(midi_output_path)
print(f"Saved MIDI to {midi_output_path}")

# Show first 10 notes
print("\nFirst 10 transcribed notes:")
print(f"{'Pitch':>6} {'Start':>8} {'End':>8} {'Velocity':>9}")
for note in midi_data.instruments[0].notes[:10]:
    print(f"{note.pitch:>6} {note.start:>8.3f} {note.end:>8.3f} {note.velocity:>9}")

In [ ]:
import pretty_midi


def plot_piano_roll(midi_data, start_time=0, end_time=None, ax=None):
    """Plot a piano roll from a PrettyMIDI object."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 5))

    notes = midi_data.instruments[0].notes

    if end_time is None:
        end_time = max(n.end for n in notes)

    for note in notes:
        if note.start >= end_time or note.end <= start_time:
            continue
        ax.barh(
            note.pitch,
            note.end - note.start,
            left=note.start,
            height=0.8,
            alpha=note.velocity / 127,
            color='steelblue',
            edgecolor='navy',
            linewidth=0.5
        )

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('MIDI Pitch')
    ax.set_xlim(start_time, end_time)

    # Add pitch labels for the range present
    pitches = [n.pitch for n in notes]
    if pitches:
        ax.set_ylim(min(pitches) - 2, max(pitches) + 2)

    return ax


# Plot full piano roll
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full view
plot_piano_roll(midi_data, ax=axes[0])
axes[0].set_title('Piano Roll — Full Transcription')

# Zoomed view (first 5 seconds)
plot_piano_roll(midi_data, start_time=0, end_time=5, ax=axes[1])
axes[1].set_title('Piano Roll — First 5 Seconds (Zoomed)')

plt.tight_layout()
plt.show()

# Also show mel spectrogram of original for comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
S_dB = librosa.power_to_db(S, ref=np.max)
librosa.display.specshow(S_dB, x_axis='time', y_axis='mel', sr=sr, ax=axes[0])
axes[0].set_title('Original Audio — Mel Spectrogram')

plot_piano_roll(midi_data, ax=axes[1])
axes[1].set_title('Transcribed MIDI — Piano Roll')

plt.tight_layout()
plt.show()

## Analysis

Automatic music transcription remains a challenging problem, especially for:

- **Polyphonic music** — multiple simultaneous voices
- **Complex timbres** — orchestral instruments vs. solo piano
- **Expressive performance** — rubato, dynamics, articulation

Below we compute some statistics about the transcription output and
discuss its quality.

In [ ]:
notes = midi_data.instruments[0].notes

# Transcription statistics
pitches = [n.pitch for n in notes]
durations = [n.end - n.start for n in notes]
velocities = [n.velocity for n in notes]

print("Transcription Statistics")
print("=" * 40)
print(f"Total notes:        {len(notes)}")
print(f"Duration:           {max(n.end for n in notes):.2f} s")
print(f"Note density:       {len(notes) / max(n.end for n in notes):.1f} notes/sec")
print(f"Pitch range:        {min(pitches)} - {max(pitches)} "
      f"({pretty_midi.note_number_to_name(min(pitches))} - "
      f"{pretty_midi.note_number_to_name(max(pitches))})")
print(f"Mean duration:      {np.mean(durations):.3f} s")
print(f"Mean velocity:      {np.mean(velocities):.0f}")

# Pitch histogram
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(pitches, bins=range(min(pitches), max(pitches) + 2),
             color='steelblue', edgecolor='navy', alpha=0.8)
axes[0].set_title('Pitch Distribution')
axes[0].set_xlabel('MIDI Pitch')
axes[0].set_ylabel('Count')

axes[1].hist(durations, bins=30, color='coral', edgecolor='darkred', alpha=0.8)
axes[1].set_title('Note Duration Distribution')
axes[1].set_xlabel('Duration (s)')
axes[1].set_ylabel('Count')

axes[2].hist(velocities, bins=20, color='mediumseagreen', edgecolor='darkgreen', alpha=0.8)
axes[2].set_title('Velocity Distribution')
axes[2].set_xlabel('Velocity')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("- Compare the piano roll with the spectrogram above.")
print("- Note density suggests polyphonic transcription quality.")
print("- Very short notes may be transcription artifacts (false positives).")
print("- Velocity variation indicates dynamic sensitivity of the model.")